# שבוע 09: ניתוח מלא — קרדומות (מטלה 4)

שיעור זה הוא בסיס המטלה הרביעית.  
ניתוח EFA מלא של קרדומות ברונזה אירופיות: PCA, סטטיסטיקה, אלומטריה.

**מטרות:**
- גרף PCA מלא עם אליפסות
- שחזור צורות ממוצעות לפי קבוצה
- מבחן MANOVA הסתברותי
- ניתוח אלומטריה (גודל vs. צורה)

In [ ]:
!pip install pyefd python-bidi -q

import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
import pyefd
rtl = get_display
print('הכל מוכן!')

In [ ]:
def parse_efa_dat(text_or_path):
    if '\n' in text_or_path:
        lines = text_or_path.strip().split('\n')
    else:
        with open(text_or_path, encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
    groups, data, sizes = [], [], []
    for line in lines[2:]:
        line = line.strip() if hasattr(line, 'strip') else line
        if not line:
            continue
        cols = line.split('\t')
        groups.append(cols[3].strip())
        sizes.append(float(cols[4].replace(',', '.').replace('E', 'e')))
        coeffs = []
        for c in cols[5:]:
            c = c.strip()
            if c and c != '-':
                try:
                    coeffs.append(float(c.replace(',', '.').replace('E', 'e')))
                except:
                    pass
        data.append(coeffs)
    min_len = min(len(d) for d in data)
    return np.array([d[:min_len] for d in data]), np.array(groups), np.array(sizes)

print('parse_efa_dat מוכן')

In [ ]:
import urllib.request

AXES_URL = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/axes/axes_efa.dat'

try:
    with urllib.request.urlopen(AXES_URL, timeout=15) as r:
        text = r.read().decode('utf-8', errors='replace')
    efa_data, groups, sizes = parse_efa_dat(text)
    print(f'נטענו {len(efa_data)} קרדומות, {efa_data.shape[1]} מקדמי EFA')
    print(f'קבוצות: {np.unique(groups, return_counts=True)}')
except Exception as e:
    print(f'שגיאה: {e} — משתמשים בנתונים סינתטיים')
    np.random.seed(42)
    base = np.random.randn(60, 120) * 0.1
    base[:30, 0] += 0.5
    base[30:, 2] += 0.4
    efa_data = base
    groups = np.array(['G3']*30 + ['G4']*30)
    sizes = np.random.uniform(50, 150, 60)
    sizes[:30] += 20

In [ ]:
from sklearn.decomposition import PCA

pca_axes = PCA()
scores_axes = pca_axes.fit_transform(efa_data)
ve_axes = pca_axes.explained_variance_ratio_ * 100
print(f'PC1: {ve_axes[0]:.1f}%, PC2: {ve_axes[1]:.1f}%')

## גרף PCA עם אליפסות — שמירה כקובץ

In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    rx, ry = np.sqrt(1 + pearson), np.sqrt(1 - pearson)
    ellipse = Ellipse((0, 0), width=rx*2, height=ry*2,
                      facecolor='none', **kwargs)
    scale_x = np.sqrt(cov[0, 0]) * n_std
    scale_y = np.sqrt(cov[1, 1]) * n_std
    transf = (transforms.Affine2D().rotate_deg(45)
               .scale(scale_x, scale_y)
               .translate(np.mean(x), np.mean(y)))
    ellipse.set_transform(transf + ax.transData)
    ax.add_patch(ellipse)

fig, ax = plt.subplots(figsize=(9, 7))
mask_g3 = groups == 'G3'
mask_g4 = groups == 'G4'

ax.scatter(scores_axes[mask_g3, 0], scores_axes[mask_g3, 1],
           c='steelblue', s=80, alpha=0.85, label=f'G3 (n={mask_g3.sum()})', zorder=3)
ax.scatter(scores_axes[mask_g4, 0], scores_axes[mask_g4, 1],
           c='darkorange', s=80, marker='s', alpha=0.85,
           label=f'G4 (n={mask_g4.sum()})', zorder=3)

confidence_ellipse(scores_axes[mask_g3, 0], scores_axes[mask_g3, 1],
                   ax, edgecolor='steelblue', lw=2, linestyle='--')
confidence_ellipse(scores_axes[mask_g4, 0], scores_axes[mask_g4, 1],
                   ax, edgecolor='darkorange', lw=2, linestyle='--')

ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.axvline(0, color='gray', lw=0.8, linestyle=':')
ax.set_xlabel(f'PC1 ({ve_axes[0]:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({ve_axes[1]:.1f}%)', fontsize=12)
ax.set_title(rtl('מרחב הצורה: קרדומות ברונזה G3 vs G4'), fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('axes_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('הגרף נשמר כ-axes_pca.png')

## שחזור צורות ממוצעות

נחשב את ממוצע המקדמים לכל קבוצה ונשחזר את המתאר.  
**API**: `pyefd.reconstruct_contour(coeffs_mat, locus=(0,0), num_points=400)`  
כאשר `coeffs_mat` הוא מטריצה בצורת `(n_harmonics, 4)`.

In [ ]:
# מספר הרמוניות
n_harm = efa_data.shape[1] // 4  # 30 הרמוניות → 120 מקדמים

mean_g3 = efa_data[mask_g3].mean(axis=0)
mean_g4 = efa_data[mask_g4].mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (title, mean_coeffs, color) in zip(axes, [
    ('G3 (ממוצע)', mean_g3, 'steelblue'),
    ('G4 (ממוצע)', mean_g4, 'darkorange')
]):
    # reshape ל-(n_harm, 4)
    coeffs_mat = mean_coeffs[:n_harm*4].reshape(n_harm, 4)
    reconstructed = pyefd.reconstruct_contour(coeffs_mat, locus=(0, 0), num_points=400)
    ax.plot(reconstructed[:, 0], reconstructed[:, 1], color=color, lw=2)
    ax.fill(reconstructed[:, 0], reconstructed[:, 1], alpha=0.2, color=color)
    ax.set_title(rtl(title), fontsize=13)
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle(rtl('צורות ממוצעות של הקרדומות לפי קבוצה'), fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def permutation_manova(X, groups, n_perm=999, seed=42):
    np.random.seed(seed)
    def f_stat(X, g):
        ug = np.unique(g)
        gm = X.mean(axis=0)
        between = sum(np.sum(g==u) * np.sum((X[g==u].mean(0) - gm)**2) for u in ug)
        within  = sum(np.sum((X[g==u] - X[g==u].mean(0))**2) for u in ug)
        return between / within if within > 0 else 0
    obs = f_stat(X, groups)
    perm = [f_stat(X, np.random.permutation(groups)) for _ in range(n_perm)]
    p = (np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1)
    return obs, p, obs / (obs + 1)

F_obs, p_val, r_sq = permutation_manova(scores_axes[:, :4], groups)
print('=== MANOVA הסתברותי — קרדומות ===')
print(f'F-statistic: {F_obs:.4f}')
print(f'p-value:     {p_val:.4f}')
print(f'R²:          {r_sq:.4f}')
if p_val < 0.05:
    print('** ההפרש בין G3 ל-G4 מובהק (p < 0.05) **')
else:
    print('אין הפרש מובהק (p >= 0.05)')

## אלומטריה: גודל vs. צורה

אלומטריה היא הקשר בין גודל לצורה.  
נבדוק: האם קרדומות גדולות יותר שונות בצורתן מקרדומות קטנות?

In [ ]:
from scipy.stats import linregress

log_sizes = np.log(sizes)
pc1_scores = scores_axes[:, 0]

slope, intercept, r_value, p_value, se = linregress(log_sizes, pc1_scores)

fig, ax = plt.subplots(figsize=(8, 6))

for mask, color, label in [(mask_g3, 'steelblue', 'G3'), (mask_g4, 'darkorange', 'G4')]:
    ax.scatter(log_sizes[mask], pc1_scores[mask],
               color=color, s=70, alpha=0.8, label=label, zorder=3)

x_line = np.linspace(log_sizes.min(), log_sizes.max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'k--', lw=2,
        label=f'קו רגרסיה (R²={r_value**2:.3f}, p={p_value:.4f})')

ax.set_xlabel(rtl('log(גודל קרדום)'), fontsize=11)
ax.set_ylabel(f'PC1 ({ve_axes[0]:.1f}%)', fontsize=11)
ax.set_title(rtl('אלומטריה: גודל קרדום vs. PC1'), fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'שיפוע: {slope:.4f}, R²: {r_value**2:.4f}, p: {p_value:.4f}')
if p_value < 0.05:
    print('** יש קשר מובהק בין גודל לצורה (אלומטריה) **')
else:
    print('אין קשר מובהק בין גודל לצורה')

## שאלות לדיון (מטלה 4)

1. **הפרדה**: האם G3 ו-G4 נפרדים בבירור? מה ה-p-value ומה משמעותו?
2. **צורות ממוצעות**: תארו בשפה ויזואלית את ההבדל בין צורת G3 לצורת G4.
3. **אלומטריה**: האם גודל הקרדום קשור לצורתו? מה ניתן להסיק?
4. **פרשנות ארכיאולוגית**: מה מלמדות הצורות השונות על ייצור, שימוש או תרבות?